# OFFLINE EVALUATION: SEARCH

In [1]:
print(123)

123


In [1]:
import json
import glob
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from embedder import Embedder
from sqlitesearch import TextSearchIndex, VectorSearchIndex

# 1. Mesclar os arquivos batch[i].json
ground_truth_dir = Path("data/ground_truth")
# Encontra todos os arquivos que começam com "batch" e terminam com ".json"
batch_files = sorted(ground_truth_dir.glob("batch*.json"))

ground_truth = []
for file_path in batch_files:
    with open(file_path, "r", encoding="utf-8") as f:
        batch_data = json.load(f)
        ground_truth.extend(batch_data)

print(f"Total de perguntas carregadas dos arquivos originais: {len(ground_truth)}")

# Salvar o arquivo consolidado
merged_path = ground_truth_dir / "ground_truth_questions.json"
with open(merged_path, "w", encoding="utf-8") as f:
    json.dump(ground_truth, f, ensure_ascii=False, indent=4)
print(f"Arquivo mesclado salvo com sucesso em: {merged_path}")

# 2. Instanciar o embedder
embedder = Embedder(path="models/Xenova/multilingual-e5-base")

# 3. Instanciar os índices de busca do sqlitesearch
DB_PATH = "mec_faq.db"

text_index = TextSearchIndex(
    text_fields=["nome", "pergunta", "resposta"],
    keyword_fields=["sigla", "agrupamento", "termos", "sinonimos"],
    id_field="doc_id",
    db_path=DB_PATH,
)

vector_index = VectorSearchIndex(
    keyword_fields=["sigla", "agrupamento", "termos", "sinonimos"],
    id_field="doc_id",
    mode="ivf",
    db_path=DB_PATH,
)

TOP_K = 5

2026-07-21 11:40:58.755072428 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


Total de perguntas carregadas dos arquivos originais: 1120
Arquivo mesclado salvo com sucesso em: data/ground_truth/ground_truth_questions.json


In [ ]:
def search_textual(pergunta, top_k=5):
    """Realiza a busca textual usando a abstração da sqlitesearch."""
    results = text_index.search(pergunta, num_results=top_k)
    return [res["doc_id"] for res in results]

def search_vectorial(query_vector, top_k=5):
    """Realiza a busca vetorial recebendo o vetor já processado."""
    results = vector_index.search(query_vector, num_results=top_k)
    return [res["doc_id"] for res in results]

def search_hybrid(pergunta, query_vector, top_k=5, rrf_k=60, weight_text=0.7, weight_vector=0.3):
    """
    Combina busca textual e vetorial usando Weighted Reciprocal Rank Fusion (Weighted RRF).
    
    Parâmetros de Peso:
    - weight_text: Ponderação da busca FTS (padrão: 0.7)
    - weight_vector: Ponderação da busca Vetorial (padrão: 0.3)
    """
    # Busca pools maiores de candidatos para enriquecer a fusão
    text_results = search_textual(pergunta, top_k=20) 
    vector_results = search_vectorial(query_vector, top_k=20)
    
    rrf_scores = {}
    
    # Pontuação RRF ponderada para busca textual
    for rank, doc_id in enumerate(text_results, 1):
        score = weight_text * (1.0 / (rrf_k + rank))
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + score
        
    # Pontuação RRF ponderada para busca vetorial
    for rank, doc_id in enumerate(vector_results, 1):
        score = weight_vector * (1.0 / (rrf_k + rank))
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + score
        
    # Ordena pelos maiores scores RRF finais e filtra os Top K
    sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return [doc_id for doc_id, score in sorted_docs[:top_k]]

# def search_textual(pergunta, top_k=5):
#     """Realiza a busca textual usando a abstração da sqlitesearch."""
#     results = text_index.search(pergunta, num_results=top_k)
#     return [res["doc_id"] for res in results]

# def search_vectorial(query_vector, top_k=5):
#     """Realiza a busca vetorial recebendo o vetor já processado."""
#     results = vector_index.search(query_vector, num_results=top_k)
#     return [res["doc_id"] for res in results]

# def search_hybrid(pergunta, query_vector, top_k=5, rrf_k=60):
#     """Combina busca textual e vetorial usando Reciprocal Rank Fusion (RRF)."""
#     # Buscamos um pool maior (num_results=20) em cada método para enriquecer a fusão
#     text_results = search_textual(pergunta, top_k=20) 
#     vector_results = search_vectorial(query_vector, top_k=20)
    
#     rrf_scores = {}
    
#     # Pontuação RRF para busca textual
#     for rank, doc_id in enumerate(text_results, 1):
#         rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (1.0 / (rrf_k + rank))
        
#     # Pontuação RRF para busca vetorial
#     for rank, doc_id in enumerate(vector_results, 1):
#         rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (1.0 / (rrf_k + rank))
        
#     # Ordena pelos maiores scores RRF e pega os Top K
#     sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
#     return [doc_id for doc_id, score in sorted_docs[:top_k]]

In [9]:
def calculate_hit_rate(retrieved_docs, expected_doc_id):
    """Retorna 1 se o documento correto foi recuperado, 0 caso contrário."""
    return 1 if expected_doc_id in retrieved_docs else 0

def calculate_mrr(retrieved_docs, expected_doc_id):
    """Calcula o Mean Reciprocal Rank para uma única consulta."""
    for rank, doc_id in enumerate(retrieved_docs, 1):
        if doc_id == expected_doc_id:
            return 1.0 / rank
    return 0.0

In [10]:
results = {
    "textual": {"hit_rate": [], "mrr": []},
    "vectorial": {"hit_rate": [], "mrr": []},
    "hybrid": {"hit_rate": [], "mrr": []}
}

BATCH_SIZE = 50 # Tamanho do lote. Pode aumentar se sua RAM permitir

# Envolvemos o range com tqdm para visualizar a barra de progresso
for i in tqdm(range(0, len(ground_truth), BATCH_SIZE), desc="Avaliando Batches"):
    # 1. Separar o lote atual
    batch = ground_truth[i : i + BATCH_SIZE]
    
    perguntas = [item["pergunta"] for item in batch]
    expected_docs = [item["doc_id"] for item in batch]
    
    # 2. Preparar os textos e gerar embeddings em LOTE
    # Prefixo "query: " é obrigatório para os modelos e5
    queries_for_embedding = [f"query: {p}" for p in perguntas]
    batch_vectors = embedder.encode_batch(queries_for_embedding)
    
    # 3. Executar as buscas para cada item dentro do lote
    for j in range(len(batch)):
        pergunta = perguntas[j]
        query_vector = batch_vectors[j]
        expected_doc_id = expected_docs[j]
        
        # Realizar as três buscas
        docs_textual = search_textual(pergunta, top_k=TOP_K)
        docs_vectorial = search_vectorial(query_vector, top_k=TOP_K)
        docs_hybrid = search_hybrid(pergunta, query_vector, top_k=TOP_K)
        
        # Registrar Textual
        results["textual"]["hit_rate"].append(calculate_hit_rate(docs_textual, expected_doc_id))
        results["textual"]["mrr"].append(calculate_mrr(docs_textual, expected_doc_id))
        
        # Registrar Vetorial
        results["vectorial"]["hit_rate"].append(calculate_hit_rate(docs_vectorial, expected_doc_id))
        results["vectorial"]["mrr"].append(calculate_mrr(docs_vectorial, expected_doc_id))
        
        # Registrar Híbrida
        results["hybrid"]["hit_rate"].append(calculate_hit_rate(docs_hybrid, expected_doc_id))
        results["hybrid"]["mrr"].append(calculate_mrr(docs_hybrid, expected_doc_id))

print("Avaliação concluída com sucesso!")

Avaliando Batches:   0%|          | 0/23 [00:00<?, ?it/s]

Avaliação concluída com sucesso!


In [11]:
# Calcular a média final de cada métrica
final_metrics = []

for search_type in ["textual", "vectorial", "hybrid"]:
    avg_hit_rate = np.mean(results[search_type]["hit_rate"])
    avg_mrr = np.mean(results[search_type]["mrr"])
    
    final_metrics.append({
        "Search Type": search_type.capitalize(),
        "Hit Rate": round(avg_hit_rate, 4),
        "Mean Reciprocal Rank (MRR)": round(avg_mrr, 4)
    })

# Criar DataFrame
df_results = pd.DataFrame(final_metrics)

# Exportar para CSV
output_csv_path = "avaliacao_busca_rag.csv"
df_results.to_csv(output_csv_path, index=False)

print(f"Resultados salvos na matriz: {output_csv_path}")
df_results

Resultados salvos na matriz: avaliacao_busca_rag.csv


,Search Type,Hit Rate,Mean Reciprocal Rank (MRR)
0,Textual,0.7205,0.5768
1,Vectorial,0.5482,0.4332
2,Hybrid,0.6848,0.5076
